# Build the bird species dataset

Interactive driver for [build_dataset.py](build_dataset.py) — all logic lives in the script, this notebook only runs its steps one
by one and shows the results. Use it for pilots and for checking the output; for full runs (e.g. on the cluster) call the script directly:

```
python build_dataset.py --workdir data_v7 [--steps ...] [options]
```

Steps: `metadata` → `split` → `download` → `frames` → `filter` → `assemble` → `report`. Every step caches its output in the workdir,
so cells can be re-run and only do new work.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

import build_dataset as bd

## Configuration
The settings are the command line options of the script (`python build_dataset.py --help` lists all of them with their defaults).
Relative paths are resolved from this folder (`data/`).

In [ ]:
args = bd.parse_args([
    "--workdir", "data_pilot",
    "--videos-per-species", "20",               # pilot: max. videos per species ("none" = all)
    "--seed-movements", "data/movements.csv",   # reuse an existing movements.csv instead of crawling the API
    "--legacy-video-dir", "data/videos",        # reuse already downloaded videos
    # "--refresh-metadata",                     # re-fetch stations and movements from the API
    # "--exclude-species", "sylvia_atricapilla", "poecile_montanus",  # drop completely (default: sylvia_atricapilla)
    # "--split-by", "video",
    # "--max-videos-per-station", "none",
    # "--min-entropy", "6.0",
    # "--filter-batch-size", "16",
])
bd.setup(args)
reports = args.workdir / "reports"

## 1. Stations and validated movements

In [ ]:
bd.run_step("metadata", args)

## 2. Classes, cap per station, split and video selection
Multi-species videos are excluded. A species gets its own class with enough videos and stations and without a dominating station;
other birds become `unknown_bird`, other non-birds are dropped. Then videos are capped per station and species, stations are assigned
to splits by the optimizer (or videos are split randomly with `--split-by video`) and the pilot sample is drawn.

In [ ]:
bd.run_step("split", args)
display(Image(reports / "support_per_species.png"))

## 3. Download videos

In [ ]:
bd.run_step("download", args)

## 4. Frame extraction
Frames that are too dark, too bright, low in contrast, low in information (entropy) or too similar to the last saved frame are skipped; the log shows how many per class and reason (also in `frame_extraction.csv`).

In [ ]:
bd.run_step("frames", args)

## 5. Bird filter
The background/unclear/bird model scores every frame; background frames become the `background` class, unclear frames are rejected.

In [ ]:
bd.run_step("filter", args)

frames = pd.read_csv(args.workdir / "frames.csv")
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([frames["p_background"], frames["p_bird"]], bins=50, label=["P(background)", "P(bird)"])
ax.axvline(args.filter_threshold, color="k", linestyle="--")
ax.set_yscale("log")
ax.set_xlabel("probability [%]")
ax.legend()
plt.show()

## 6. Assemble `dataset/<split>/<class>/`

In [ ]:
bd.run_step("assemble", args)

## 7. Report
Images, videos and stations per class and split, leakage checks and sample frames per class (also saved to `reports/`).

In [ ]:
bd.run_step("report", args)
display(Image(reports / "samples_train.png"))